In [ ]:
import pandas as pd
from pandas.api.types import is_string_dtype

def format_ship_dataset(df, 
                        survived_col,   # nome colonna sopravvivenza nel dataset
                        sex_col,        # nome colonna sesso
                        age_col,        # nome colonna età
                        class_col=None, # nome colonna classe 
                        crew_col=None,  # nome colonna Crew/Passanger
                        #crew_prob=0.15  # probabilità di essere Crew se generiamo casuale
                       ):
    """
    Trasforma qualsiasi dataset navale in un formato uniforme:
    ['survived', 'male', 'female', 'age', 'class', 'crew']
    """
    
    df_new = pd.DataFrame()

    # --- Sopravvivenza ---
    if is_string_dtype(df[survived_col]):
        # Se dtype è string allora converto a numero
        df_new['survived'] = df[survived_col].str.lower().map({'lost': 0, 'saved': 1})
    else:
        df_new['survived'] = df[survived_col].astype(int)

    # --- Sesso ---
    # un solo attributo per il sesso e non due di cui uno derivabile dal primo (female = 1 - male)
    df_new['sex'] = df[sex_col].fillna("").str.lower().isin(['male', 'm']).astype(int) 
    #df_new['male'] = df[sex_col].str.lower().isin(['male', 'm']).astype(int)
    #df_new['female'] = df[sex_col].str.lower().isin(['female', 'f']).astype(int)

    # --- Età ---
    median_age = round(df[age_col].median()) # se l'età non c'è si prende l'età mediana
    df_new['age'] = df[age_col].fillna(median_age)

    df_new["age2"] = df_new["age"] ** 2 # età al quadrato per considerare di più i passegeri anziani

    # --- Classe ---
    if class_col is not None:
        df_new['class'] = df[class_col].fillna(3).astype(int)
    else:
        # Generiamo class casuali con distribuzione 3=50%, 2=35%, 1=15%
        #df_new['class'] = np.random.choice([3,2,1], size=len(df_new), p=[0.5, 0.35, 0.15])
        df_new['class'] = 3 # meglio default di casuale

    df_new["sex_class"] = df_new["sex"] * df_new["class"] # considerare la classe in relazione al sesso può aiutare come donne in prima classe

    # --- Crew ---
    if crew_col is not None:
        df_new['crew'] = df[crew_col].map({'P': 0, 'Passenger': 0, 'C': 1, 'Crew': 1}).fillna(0).astype(int)
    else:
        # Generiamo colonna crew casuale 0/1 con crew_prob
        #df_new['crew'] = np.random.choice([0,1], size=len(df_new), p=[1-crew_prob, crew_prob])
        df_new['crew'] = 0 # meglio default di casuale

    return df_new


In [10]:
# Caricamento dei CSV
df_titanic = pd.read_csv("../data/titanic.csv")
df_estonia = pd.read_csv("../data/estonia.csv")
df_lusitania = pd.read_csv("../data/lusitania.csv")

print(f"Titanic: {df_titanic.shape}")
print(f"Estonia: {df_estonia.shape}")
print(f"Lusitania: {df_lusitania.shape}")

Titanic: (891, 12)
Estonia: (989, 8)
Lusitania: (1961, 16)


In [11]:
# TITANIC FORMATTATO
df_titanic_formatted = format_ship_dataset(df_titanic,
                                           survived_col='Survived',
                                           sex_col='Sex',
                                           age_col='Age',
                                           class_col='Pclass')
# ESTONIA FORMATTATO
df_estonia_formatted = format_ship_dataset(df_estonia,
                                           survived_col='Survived',
                                           sex_col='Sex',
                                           age_col='Age',
                                           crew_col='Category')
# LUSITANIA FORMATTATO
df_lusitania_formatted = format_ship_dataset(df_lusitania,
                                             survived_col='Fate',
                                             sex_col='Sex',
                                             age_col='Age',
                                             crew_col='Passenger/Crew')


print(df_titanic_formatted.head(5))
print(df_estonia_formatted.head(5))
print(df_lusitania_formatted.head(5))

   survived  sex   age    age2  class  sex_class  crew
0         0    1  22.0   484.0      3          3     0
1         1    0  38.0  1444.0      1          0     0
2         1    0  26.0   676.0      3          0     0
3         1    0  35.0  1225.0      1          0     0
4         0    1  35.0  1225.0      3          3     0
   survived  sex  age  age2  class  sex_class  crew
0         0    1   62  3844      3          3     0
1         0    0   22   484      3          0     1
2         0    0   21   441      3          0     1
3         0    1   53  2809      3          3     1
4         0    0   55  3025      3          0     0
   survived  sex   age    age2  class  sex_class  crew
0         0    1  38.0  1444.0      3          3     1
1         0    1  37.0  1369.0      3          3     1
2         1    1  30.0   900.0      3          3     1
3         1    1  25.0   625.0      3          3     1
4         1    1  27.0   729.0      3          3     1


In [12]:

# Uniamo tutti e tre i dataset
all_ships = pd.concat([df_titanic_formatted, df_lusitania_formatted, df_estonia_formatted], ignore_index=True)
print(all_ships)

all_ships.to_csv("../data/all_ships.csv", index=False)


      survived  sex   age    age2  class  sex_class  crew
0            0    1  22.0   484.0      3          3     0
1            1    0  38.0  1444.0      1          0     0
2            1    0  26.0   676.0      3          0     0
3            1    0  35.0  1225.0      1          0     0
4            0    1  35.0  1225.0      3          3     0
...        ...  ...   ...     ...    ...        ...   ...
3836         0    0  60.0  3600.0      3          0     0
3837         1    1  34.0  1156.0      3          3     0
3838         0    1  77.0  5929.0      3          3     0
3839         0    0  87.0  7569.0      3          0     0
3840         1    1  42.0  1764.0      3          3     0

[3841 rows x 7 columns]
